<a href="https://colab.research.google.com/github/njwbilll/Tugas-5_Grokking-Deep-Learning-MANNING_Najwa-Bilqis-Al-Khalidah/blob/main/10_Intro_to_Convolutional_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10: Saraf Pembelajar Tepi dan Sudut Pengantar Jaringan Konvolusional

**Referensi:** Grokking Deep Learning, Andrew W. Trask (Manning Publications, 2019)

## Ringkasan Bab

Bab kesepuluh membuka gerbang menuju arsitektur paling revolusioner dalam bidang visi komputer yaitu Jaringan Saraf Konvolusional. Jika sebelumnya kita meratakan seluruh piksel gambar menjadi satu barisan panjang secara mentah, metode tersebut menghilangkan informasi spasial yang sangat berharga. Bab ini mengajarkan cara merawat dimensi asli gambar dan memindainya menggunakan filter pencari kecil yang bergerak menyapu seluruh permukaan kanvas visual. Teknik ini tidak hanya mempertahankan konteks posisi dan bentuk geometri, namun juga menghemat jutaan parameter secara drastis melalui konsep pembagian beban bobot.

## Pemahaman Teori Konvolusi dan Penggunaan Ulang Bobot

### A. Masalah Fundamental Pada Jaringan Padat
Jika kita memiliki gambar resolusi tinggi, menghubungkan setiap piksel secara individual ke setiap perantara pada lapisan selanjutnya akan membutuhkan miliaran parameter. Hal ini memicu kesulitan belajar dan pemborosan memori komputasi yang sangat luar biasa.

### B. Konsep Cerdas Penggunaan Ulang Bobot
Sebagai solusi brilian, kita menciptakan sebuah matriks parameter berukuran sangat kecil, misalnya tiga kali tiga piksel. Matriks inilah yang disebut sebagai saringan atau kernel. Alih alih membuat parameter baru untuk setiap sudut gambar, kita mendaur ulang saringan yang sama persis dan menggesernya selangkah demi selangkah ke seluruh penjuru kanvas data.

### C. Pencarian Tepi Sudut dan Pola
Pada tahap awal pelatihan, kernel akan berevolusi secara alami untuk mendeteksi pola paling mendasar seperti garis vertikal, garis horizontal, atau transisi gelap terang. Deteksi level rendah ini kemudian menjadi fondasi kokoh bagi lapisan selanjutnya untuk merangkai bentuk yang jauh lebih rumit menyerupai wujud benda asli.

### Tahap Persiapan Pustaka Utilitas
Kita memuat modul operasional aljabar untuk mengatur pembentukan susunan matriks spasial dimensi tinggi.

In [1]:
import numpy as np
print("Modul komputasi spasial sukses bersiaga untuk memproses citra dua dimensi")

Modul komputasi spasial sukses bersiaga untuk memproses citra dua dimensi


### Deklarasi Fungsi Aktivasi Pembengkok Ruang
Kita menghadirkan kembali fungsi pemangkas relu agar sistem sanggup membedakan pola yang melengkung dan tidak selamanya linear tegak lurus.

In [2]:
def aktivasi_relu(nilai_matriks):
    return np.multiply((nilai_matriks > 0), nilai_matriks)

def kemiringan_relu(nilai_matriks):
    return nilai_matriks > 0

print("Katalisator kelenturan logika telah disiapkan sepenuhnya")

Katalisator kelenturan logika telah disiapkan sepenuhnya


### Mekanisme Propagasi Maju Konvolusional
Fungsi di bawah ini merumuskan cara kerja sebuah filter pemindai. Algoritma akan memotong area kecil dari gambar asli yang dimensinya ekuivalen dengan kernel, melakukan perkalian titik secara internal, menjumlahkan hasilnya, lalu mencatatnya sebagai satu titik kebenaran baru pada peta fitur keluaran.

In [3]:
def konvolusi_maju(citra_input, kernel_filter):
    baris_citra = citra_input.shape[0]
    kolom_citra = citra_input.shape[1]
    baris_kernel = kernel_filter.shape[0]
    kolom_kernel = kernel_filter.shape[1]

    batas_baris = np.add(np.subtract(baris_citra, baris_kernel), 1)
    batas_kolom = np.add(np.subtract(kolom_citra, kolom_kernel), 1)

    hasil_konvolusi = np.zeros((batas_baris, batas_kolom))

    for indeks_baris in range(batas_baris):
        for indeks_kolom in range(batas_kolom):
            akhir_baris = np.add(indeks_baris, baris_kernel)
            akhir_kolom = np.add(indeks_kolom, kolom_kernel)

            potongan_citra = citra_input[indeks_baris:akhir_baris, indeks_kolom:akhir_kolom]
            hasil_konvolusi[indeks_baris, indeks_kolom] = np.sum(np.multiply(potongan_citra, kernel_filter))

    return hasil_konvolusi

print("Mesin peraba dimensi gambar sukses diinisialisasi")

Mesin peraba dimensi gambar sukses diinisialisasi


### Mekanisme Propagasi Mundur Konvolusional
Kunci paling esensial dari pembaruan kecerdasan dalam arsitektur ini adalah mendistribusikan galat kembali kepada kernel induk. Karena kernel yang sama digunakan berkali kali di berbagai posisi koordinat, maka gradien koreksi final untuk kernel tersebut didapat dari hasil penjumlahan total seluruh koreksi di setiap posisi pindaian.

In [4]:
def konvolusi_mundur(citra_input, gradien_lapisan_selanjutnya, ukuran_kernel):
    baris_kernel = ukuran_kernel[0]
    kolom_kernel = ukuran_kernel[1]

    gradien_untuk_kernel = np.zeros(ukuran_kernel)

    batas_baris = gradien_lapisan_selanjutnya.shape[0]
    batas_kolom = gradien_lapisan_selanjutnya.shape[1]

    for indeks_baris in range(batas_baris):
        for indeks_kolom in range(batas_kolom):
            akhir_baris = np.add(indeks_baris, baris_kernel)
            akhir_kolom = np.add(indeks_kolom, kolom_kernel)

            potongan_citra = citra_input[indeks_baris:akhir_baris, indeks_kolom:akhir_kolom]
            nilai_gradien_titik = gradien_lapisan_selanjutnya[indeks_baris, indeks_kolom]

            perubahan_sebagian = np.multiply(potongan_citra, nilai_gradien_titik)
            gradien_untuk_kernel = np.add(gradien_untuk_kernel, perubahan_sebagian)

    return gradien_untuk_kernel

print("Algoritma distribusi kesalahan spasial siap diluncurkan")

Algoritma distribusi kesalahan spasial siap diluncurkan


### Konstruksi Ekosistem Simulasi Citra Sintetis
Kita menciptakan ratusan kanvas persegi berukuran delapan kali delapan matriks piksel sebagai wadah latihan. Data tersebut dijodohkan bersama sebuah kumpulan label sasaran biner absolut guna memvalidasi ketangguhan pembelajar.

In [5]:
np.random.seed(888)
jumlah_sampel = 150
dimensi_kanvas = 8

kumpulan_citra = np.random.rand(jumlah_sampel, dimensi_kanvas, dimensi_kanvas)
kumpulan_label = np.random.randint(2, size=(jumlah_sampel, 1))

print("Tinjauan proporsi bongkahan data visual:", kumpulan_citra.shape)
print("Tinjauan proporsi kebenaran hakiki:", kumpulan_label.shape)

Tinjauan proporsi bongkahan data visual: (150, 8, 8)
Tinjauan proporsi kebenaran hakiki: (150, 1)


### Inisialisasi Memori Otak Terhindar Dari Simbol Terlarang
Kita memahat blok ingatan konvolusi awal beserta matriks padat pengambil konklusi. Seluruh bilangan yang harus menyentuh area bawah nol diformulasikan murni menggunakan teknik substitusi fungsi pengurangan aljabar numerik.

In [6]:
acak_filter = np.multiply(np.random.rand(3, 3), 2.0)
kernel_utama = np.subtract(acak_filter, 1.0)

acak_padat = np.multiply(np.random.rand(36, 1), 2.0)
bobot_pengambil_keputusan = np.subtract(acak_padat, 1.0)

laju_adaptasi = 0.005
print("Dua modul kecerdasan spasial sukses dilahirkan dalam kondisi sempurna")

Dua modul kecerdasan spasial sukses dilahirkan dalam kondisi sempurna


### Orkestrasi Siklus Hidup Jaringan Saraf Konvolusional
Di fase pembuktian ini, kita merangkai gerak maju pemindai bidang spasial, pembengkokan batas akurasi melalui fungsi relu, pemipihan dimensi data, hingga konklusi akhir. Begitu selisih simpangan terdeteksi, gelombang kesalahan akan membanjir mundur mencetak ulang karakter saringan hingga model meraih pencerahan wujud aslinya.

In [7]:
for fase_pembelajaran in range(30):
    total_penyimpangan_epos = 0.0

    for posisi_data in range(jumlah_sampel):
        citra_aktif = kumpulan_citra[posisi_data]
        label_aktif = kumpulan_label[posisi_data]

        peta_analisis = konvolusi_maju(citra_aktif, kernel_utama)
        peta_teraktivasi = aktivasi_relu(peta_analisis)

        vektor_perataan = peta_teraktivasi.reshape((1, 36))
        kesimpulan_mentah = np.dot(vektor_perataan, bobot_pengambil_keputusan)

        jarak_kebenaran = np.subtract(kesimpulan_mentah, label_aktif)
        total_penyimpangan_epos = np.add(total_penyimpangan_epos, np.power(jarak_kebenaran, 2)[0, 0])

        panduan_padat = np.multiply(vektor_perataan.T, jarak_kebenaran)

        panduan_transisi_datar = np.multiply(bobot_pengambil_keputusan, jarak_kebenaran).T
        panduan_transisi_ruang = panduan_transisi_datar.reshape((6, 6))

        panduan_sebelum_relu = np.multiply(panduan_transisi_ruang, kemiringan_relu(peta_analisis))

        revisi_untuk_kernel = konvolusi_mundur(citra_aktif, panduan_sebelum_relu, (3, 3))

        koreksi_padat = np.multiply(laju_adaptasi, panduan_padat)
        bobot_pengambil_keputusan = np.subtract(bobot_pengambil_keputusan, koreksi_padat)

        koreksi_kernel = np.multiply(laju_adaptasi, revisi_untuk_kernel)
        kernel_utama = np.subtract(kernel_utama, koreksi_kernel)

    if (np.add(fase_pembelajaran, 1) % 5) == 0:
        status_putaran = np.add(fase_pembelajaran, 1)
        print("Penyelesaian putaran", status_putaran, "merekam jejak error kumulatif di titik", total_penyimpangan_epos)

print("\nSimulasi komprehensif ekstraksi gambar berbasis jaringan saringan sukses dieksekusi tuntas.")

Penyelesaian putaran 5 merekam jejak error kumulatif di titik 43.164291745254644
Penyelesaian putaran 10 merekam jejak error kumulatif di titik 40.46122157178458
Penyelesaian putaran 15 merekam jejak error kumulatif di titik 39.915692388667466
Penyelesaian putaran 20 merekam jejak error kumulatif di titik 39.63700951554438
Penyelesaian putaran 25 merekam jejak error kumulatif di titik 39.414303026003715
Penyelesaian putaran 30 merekam jejak error kumulatif di titik 39.20709883341883

Simulasi komprehensif ekstraksi gambar berbasis jaringan saringan sukses dieksekusi tuntas.
